In [1]:
print("hi")

hi


In [ ]:
import json
import re
import os
from typing import List
from PIL import Image
from pydantic import BaseModel
import pdfplumber
# from pipe_fn import pipe
import torch
import cv2
import numpy as np
from output_utils import save_split_output


from transformers import pipeline


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)




# =========================
# CROP PT → REFERENCE
# =========================
def crop_all_eob_tables(pdf_path, output_dir="cropped_tables"):
    pdf_dir=os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)

    output_dir=os.path.join(output_dir,pdf_dir)
    os.makedirs(output_dir, exist_ok=True)

    res = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(
            pdf.pages,
            start=1
        ):

            print(f"\n🧾 Cropping Page {page_num}")

            start_positions = []
            end_positions = []

            start_hits = page.search(
                "Claim #"
            )

            end_hits = page.search(
                "Claim Sub-Totals"
            )

            if not start_hits:

                start_hits = page.search(
                    "Claim #"
                )

            if not end_hits:

                end_hits = page.search(
                    "Claim Sub-Totals"
                )

            for hit in start_hits:

                start_positions.append(
                    hit["top"] - 8
                )

            for hit in end_hits:

                end_positions.append(
                    hit["bottom"] + 10
                )

            table_count = min(
                len(start_positions),
                len(end_positions)
            )

            if table_count == 0:

                print("❌ No tables found")

                continue

            for idx in range(table_count):

                bbox = (
                    0,
                    start_positions[idx],
                    page.width,
                    end_positions[idx]
                )

                cropped_page = page.crop(
                    bbox
                )

                output_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )

                cropped_page.to_image(
                    resolution=300
                ).save(output_path)

                expected_rows = count_service_rows(
                    page,
                    start_positions[idx],
                    end_positions[idx]
                )

                print(f"✅ Saved: {output_path}")
                print(f"📊 Expected rows: {expected_rows}")




                res.append({
                    "page": page_num,
                    "table": idx + 1,
                    "output_path": output_path,
                    "expected_rows": expected_rows
                })

    return res

def save_images(claims, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    paths = []

    for claim in claims:
        paths.append({
            "image_path": claim["output_path"],
            "expected_rows": claim.get("expected_rows", 0)
        })

    return paths


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    sums = np.sum(thresh, axis=1)
    th = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]

    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)

    return img

def convert_amounts_to_string(obj):

    amount_fields = {
        "total_charges",
        "charge_reduction",
        "non_covered_services",
        "deductible",
        "coinsurance_copay",
        "other_plan",
        "benefits_paid"
    }

    if isinstance(obj, dict):

        new_obj = {}

        for k, v in obj.items():

            if k in amount_fields:

                try:

                    clean_value = (
                        str(v)
                        .replace("$", "")
                        .replace(",", "")
                        .strip()
                    )

                    new_obj[k] = f"{float(clean_value):.2f}"

                except:
                    new_obj[k] = ""

            else:
                new_obj[k] = convert_amounts_to_string(v)

        return new_obj

    elif isinstance(obj, list):

        return [convert_amounts_to_string(i) for i in obj]

    return obj

def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    service_rows = set()

    for w in words:

        text = w["text"].strip()
        y = round(float(w["top"]), 1)

        if region_top <= y <= region_bottom:

            # Charge numbers: 01,02,03...
            if re.fullmatch(r"\d{2}", text):
                service_rows.add(y)

    return len(service_rows)


# =========================
# PROMPT
# =========================
def build_prompt(pdf_name):

    return """
You are extracting structured financial data from a dental EOB image.

STRICT RULES

OUTPUT ONLY VALID JSON.

DO NOT WRITE EXPLANATIONS.

DO NOT HALLUCINATE.

DO NOT CALCULATE VALUES.

DO NOT REMOVE DUPLICATE ROWS.

IF TWO ROWS LOOK IDENTICAL,
EXTRACT BOTH ROWS.

--------------------------------
COLUMN RULES
--------------------------------
provider
-> ONLY from the Provider: from the header of the table.

date_of_service
→ ONLY from Date(s) of Service


total_charges
→ ONLY from Total Charges

charge_reduction
→ ONLY from Charge Reduction

non_covered_services
→ ONLY from Non-Covered Services

deductible
→ ONLY from Allowed

coinsurance_copay
→ ONLY from Coinsurance/Co-pay

other_plan
→ ONLY from Other Plan

benefits_paid
→ ONLY from Benefits Paid



--------------------------------
ROW RULES
--------------------------------

A SERVICE ROW EXISTS ONLY IF:

date_of_service exists


If either is missing:
DO NOT create a service row.

--------------------------------
CLAIM TOTALS
--------------------------------

Claim Sub-Totals is NOT a service row.

Extract separately.

Do not calculate totals.

Extract only what appears in the Claim Sub-Totals row.

--------------------------------
OUTPUT
--------------------------------

{
  "patient_name": {
    "value": "",
    "confidence": 0.0
  },

  "provider": {
    "value": "",
    "confidence": 0.0
  },

  "date_of_service": {
    "value": "",
    "confidence": 0.0
  },

  "services": [
    {
      "total_charges": {
        "value": "",
        "confidence": 0.0
      },

      "charge_reduction": {
        "value": "",
        "confidence": 0.0
      },

      "non_covered_services": {
        "value": "",
        "confidence": 0.0
      },

      "deductible": {
        "value": "",
        "confidence": 0.0
      },

      "coinsurance_copay": {
        "value": "",
        "confidence": 0.0
      },

      "other_plan": {
        "value": "",
        "confidence": 0.0
      },

      "benefits_paid": {
        "value": "",
        "confidence": 0.0
      }
    }
  ],

  "claim_totals": {
    "total_charges": {
      "value": "",
      "confidence": 0.0
    },

    "charge_reduction": {
      "value": "",
      "confidence": 0.0
    },

    "non_covered_services": {
      "value": "",
      "confidence": 0.0
    },

    "deductible": {
      "value": "",
      "confidence": 0.0
    },

    "coinsurance_copay": {
      "value": "",
      "confidence": 0.0
    },

    "other_plan": {
      "value": "",
      "confidence": 0.0
    },

    "benefits_paid": {
      "value": "",
      "confidence": 0.0
    }
  }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.


"""
def parse_amount(x):
    if x is None or x == "":
        return 0.0
    return float(
        str(x)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )
def compute_totals_from_services(services):
    return {
        "total_charges": round(sum(parse_amount(s.get("total_charges", "")) for s in services), 2),
        "charge_reduction": round(sum(parse_amount(s.get("charge_reduction", "")) for s in services), 2),
        "non_covered_services": round(sum(parse_amount(s.get("non_covered_services", "")) for s in services), 2),
        "deductible": round(sum(parse_amount(s.get("deductible", "")) for s in services), 2),
        "coinsurance_copay": round(sum(parse_amount(s.get("coinsurance_copay", "")) for s in services), 2),
        "other_plan": round(sum(parse_amount(s.get("other_plan", "")) for s in services), 2),
        "benefits_paid": round(sum(parse_amount(s.get("benefits_paid", "")) for s in services), 2)
    }


def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):

    services = patient.get("services", [])
    totals = patient.get("claim_totals",{})

    if not services:
        return False, "No services found", [{"error": "empty services"}]

    computed_totals = compute_totals_from_services(services)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    # ===== FIELD VALIDATION =====
    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01   # 🔥 tolerance fix

        if match:
            icon = "✅"
            status = "MATCH"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    # ===== ROW COUNT VALIDATION =====
    extracted_row_count = len(services)

    if expected_row_count == extracted_row_count:
        icon = "✅"
        status = "MATCH"
    else:
        icon = "❌"
        status = "MISMATCH"
        has_error = True

        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line

    print("-" * 80)

    # ===== FINAL STATUS =====
    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, []

# =========================
# JSON CLEANER
# =========================
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1
    return json.loads(text[start:end])

def save_json(data, output_path):
    with open(output_path, "w", encoding = "utf-8") as f:
        json.dump(data, f, indent = 2, ensure_ascii = False)

def check_claim_denied(pdf_path):

    denial_keywords = [
        "denied",
        "denial"
    ]

    stop_word = "About Your Rights"

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            full_text = page.extract_text()

            if not full_text:
                continue

            searchable_text = full_text

            # Stop searching after About Your Rights
            stop_idx = searchable_text.find(stop_word)

            if stop_idx != -1:
                searchable_text = searchable_text[:stop_idx]

            for keyword in denial_keywords:

                if keyword in searchable_text:

                    print(
                        f"❌ Claim denied keyword found: "
                        f"'{keyword}' on page {page_num}"
                    )

                    return "denied"

    return "not denied"


def normalize_service_dates(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ("service_date", "service_dates") and isinstance(v, str):
                v = v.strip()

                # If two dates are separated by '-', keep only the second part
                if "-" in v:
                    left, right = v.split("-", 1)

                    # Case: MM/DD-MM/DD/YYYY
                    if re.fullmatch(r"\d{2}/\d{2}", left) and re.fullmatch(r"\d{2}/\d{2}/\d{4}", right):
                        obj[k] = right

                    # Case: MM/DD/YY-MM/DD/YY
                    elif re.fullmatch(r"\d{2}/\d{2}/\d{2}", left) and re.fullmatch(r"\d{2}/\d{2}/\d{2}", right):
                        obj[k] = left
            else:
                normalize_service_dates(v)

    elif isinstance(obj, list):
        for item in obj:
            normalize_service_dates(item)

    return obj

def run_pipeline(pdf_path, output_dir="EOB_OUTPUT/Group_administration", company_name="Group Administrator"):

    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path) 

    base_dir = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")

    os.makedirs(base_dir, exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_paths = crop_all_eob_tables(
                pdf_path,
                output_dir=cropped_dir
            )

    final_prompt = build_prompt(pdf_name)
    is_denied = check_claim_denied(pdf_path)
    print(f"claim status :{is_denied}")

    results = []

    for idx, item in enumerate(image_paths):
        img_path = item["output_path"]
        expected_rows = item.get("expected_rows", 0)

        print(f"Processing {idx+1}/{len(image_paths)}")

        image = make_table(img_path)
        image = Image.fromarray(image).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt}
                ]
            }
        ]

        with torch.no_grad():
            output = pipe(messages, max_new_tokens=5000, temperature= 0.0, do_sample=False)

        raw = output[0]["generated_text"]
        print("this is raw output", raw )

        if isinstance(raw, list):
            raw = raw[-1]["content"]


        try:
            parsed = extract_json(raw)

            model_confidence = calculate_model_confidence(parsed)
            parsed = _unwrap_vlm_output(parsed)
            parsed["_model_confidence"] = model_confidence

            if isinstance(parsed.get("date_of_service"), list):
                    parsed["date_of_service"] = (
                        parsed["date_of_service"][0]
                        if parsed["date_of_service"]
                        else ""
                    )

            parsed = convert_amounts_to_string(parsed)
            parsed = normalize_service_dates(parsed)
            results.append(parsed)
            parsed["_expected_rows"] = expected_rows
            print("✔ extracted")
        except Exception as e:
            print(f"❌ json failed:{e}")

        for patient in results:

            is_valid, log, errors = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", ""),
            expected_row_count=patient.get("_expected_rows", 0)  # or your external expected count
        )

            patient["validation"] = {
                "status": is_valid,
                "errors": errors
            }
    confidence_score = calculate_eob_confidence(results)   # ADD — before popping temp keys

    for patient in results:
        patient.pop("_expected_rows", None)
        patient.pop("_model_confidence", None)   # ADD

    final =[
        {
        "eob_id": pdf_name,
        "file_name": pdf_full_name,    
        "claim_status": is_denied,
        "payor": "Group Administrators",
        "confidence_score": confidence_score,  # ADD
        "human_review":True,
        "patients": results
        }
    ]

    success_path, failed_path = save_split_output(
                            final,
                            company_name=company_name,
                            pdf_name=pdf_name,
                            pdf_path=pdf_path,
                            cropped_dir=cropped_dir,
                        )
                    
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 19:05:30.339000 3502250 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:05:30.353000 3502250 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Group Administration/pdf's/Pmt_EOP_508188116.pdf")


🧾 Cropping Page 1
✅ Saved: EOB_OUTPUT/Group_administration/508188116/cropped_images/508188116/page_1_table_1.png
📊 Expected rows: 6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


claim status :not denied
Processing 1/1
this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=2550x615 at 0x77C9A32C7EC0>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n--------------------------------\nCOLUMN RULES\n--------------------------------\nprovider\n-> ONLY from the Provider: from the header of the table.\n\ndate_of_service\n→ ONLY from Date(s) of Service\n\n\ntotal_charges\n→ ONLY from Total Charges\n\ncharge_reduction\n→ ONLY from Charge Reduction\n\nnon_covered_services\n→ ONLY from Non-Covered Services\n\ndeductible\n→ ONLY from Allowed\n\ncoinsurance_copay\n→ ONLY from Coinsurance/Co-pay\n\nother_plan\n→ ONLY from Other Plan\n\nbenefits_paid\n→ ONLY fro

[{'eob_id': '508188116',
  'claim_status': 'not denied',
  'payor': 'Group Administrators',
  'confidence_score': 100.0,
  'human_review': True,
  'patients': [{'patient_name': 'FRANCIS KUCZYNSKI',
    'provider': 'DUC TANG DDS PLLC',
    'date_of_service': '05/07-05/07/2024',
    'services': [{'total_charges': '37.00',
      'charge_reduction': '0.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '0.00',
      'other_plan': '0.00',
      'benefits_paid': '37.00'},
     {'total_charges': '20.00',
      'charge_reduction': '0.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '0.00',
      'other_plan': '0.00',
      'benefits_paid': '20.00'},
     {'total_charges': '16.00',
      'charge_reduction': '0.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '0.00',
      'other_plan': '0.00',
      'benefits_paid': '16.00'},
     {'total_charges': '48

In [3]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Group Administration/pdf's/Pmt_EOP_600922358.pdf")


🧾 Cropping Page 1
✅ Saved: EOB_OUTPUT/Group_administration/600922358/cropped_images/600922358/page_1_table_1.png
📊 Expected rows: 5


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


claim status :not denied
Processing 1/1
this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=2550x568 at 0x73841D6E3980>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n--------------------------------\nCOLUMN RULES\n--------------------------------\nprovider\n-> ONLY from the Provider: from the header of the table.\n\ndate_of_service\n→ ONLY from Date(s) of Service\n\n\ntotal_charges\n→ ONLY from Total Charges\n\ncharge_reduction\n→ ONLY from Charge Reduction\n\nnon_covered_services\n→ ONLY from Non-Covered Services\n\ndeductible\n→ ONLY from Allowed\n\ncoinsurance_copay\n→ ONLY from Coinsurance/Co-pay\n\nother_plan\n→ ONLY from Other Plan\n\nbenefits_paid\n→ ONLY fro

[{'eob_id': '600922358',
  'claim_status': 'not denied',
  'payor': 'Group Administrators',
  'confidence_score': 100.0,
  'human_review': True,
  'patients': [{'patient_name': 'FRANCIS KUCZYNSKI',
    'provider': 'DUC TANG DDS PLLC',
    'date_of_service': '12/05-12/05/2024',
    'services': [{'total_charges': '120.00',
      'charge_reduction': '0.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '24.00',
      'other_plan': '0.00',
      'benefits_paid': '96.00'},
     {'total_charges': '920.00',
      'charge_reduction': '626.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '58.80',
      'other_plan': '0.00',
      'benefits_paid': '235.20'},
     {'total_charges': '109.00',
      'charge_reduction': '0.00',
      'non_covered_services': '0.00',
      'deductible': '0.00',
      'coinsurance_copay': '21.80',
      'other_plan': '0.00',
      'benefits_paid': '87.20'},
     {'total_char